# 📊 Step 2 — Technical Indicators (Exact Zerodha Match)

**How Zerodha/TradingView actually calculates these:**

The **Smoothing Line** setting in Zerodha/TradingView is a **separate signal line**
plotted alongside the indicator — it does NOT change the indicator value itself.
The indicator value plotted on the chart is always the pure calculation.

| Indicator | Zerodha Value (what the number shows) | Smoothing Line (separate visual line only) |
|---|---|---|
| EMA 20 | Pure `EMA(close, 20)` — no modification | SMA(9) of EMA plotted separately |
| EMA 50 | Pure `EMA(close, 50)` — no modification | SMA(9) of EMA plotted separately |
| RSI 14 | Pure `RSI(close, 14)` using Wilder's RMA | SMA(14) of RSI plotted separately |
| BB | `SMA(20) ± 2×std` | No smoothing |
| MACD | `EMA(12) − EMA(26)` | Signal = `EMA(9)` of MACD |

⚠️ **RSI in TradingView uses `ta.rsi()` which internally uses Wilder's RMA (not simple rolling mean)**

---
### Instructions
1. Run **Cell 1** — install libraries
2. Run **Cell 2** — upload your CSV from Step 1
3. Run **Cells 3 to 8** — calculate each indicator
4. Run **Cell 9** — full preview
5. Run **Cell 10** — save and download

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install pandas numpy openpyxl -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Upload and Load OHLC CSV File
# ─────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from google.colab import files

print('Please upload your Nifty 50 CSV file (from Step 1)...')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'\n✓ File uploaded: {filename}')

df = pd.read_csv(filename)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

for col in ['Open', 'High', 'Low', 'Close']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=['Open', 'High', 'Low', 'Close'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'✓ Rows loaded  : {len(df)} trading days')
print(f'✓ Date range   : {df["Date"].iloc[0].date()} → {df["Date"].iloc[-1].date()}')
print(f'\nFirst 3 rows:')
print(df.head(3).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — EMA 20 (Exact Zerodha match)
#
# Zerodha plots: pure EMA(close, 20)
# Formula      : alpha = 2 / (20 + 1)
#                EMA_t = alpha × Close_t + (1 − alpha) × EMA_(t-1)
# Seed value   : first EMA value = first Close price (same as TradingView)
#
# The "Smoothing Line SMA(9)" shown in Zerodha settings is a SEPARATE
# signal line plotted on the chart — it does NOT change the EMA value.
# ─────────────────────────────────────────────────────────────────────────
def calc_ema(series, period):
    """
    Pure EMA matching TradingView ta.ema() exactly.
    Seed: first value = first price (not SMA seed)
    alpha = 2 / (period + 1)
    """
    alpha  = 2 / (period + 1)
    result = [None] * len(series)
    for i, val in enumerate(series):
        if i == 0:
            result[i] = val           # seed = first price value
        else:
            result[i] = alpha * val + (1 - alpha) * result[i - 1]
    return pd.Series(result, index=series.index)

df['EMA_20'] = calc_ema(df['Close'], 20).round(2)

print('✓ EMA_20 calculated — exact Zerodha match')
print(f'  Formula : alpha × Close + (1-alpha) × prev_EMA')
print(f'  alpha   : 2 / (20+1) = {2/21:.6f}')
print(f'  Seed    : first EMA = first Close price (TradingView method)')
print(f'  Current : {df["EMA_20"].iloc[-1]}')
print(f'\n  Sample (last 5 rows):')
print(df[['Date', 'Close', 'EMA_20']].tail(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — EMA 50 (Exact Zerodha match)
#
# Zerodha plots: pure EMA(close, 50)
# Formula      : alpha = 2 / (50 + 1)
#                EMA_t = alpha × Close_t + (1 − alpha) × EMA_(t-1)
# ─────────────────────────────────────────────────────────────────────────
df['EMA_50'] = calc_ema(df['Close'], 50).round(2)

# EMA crossover signal
df['EMA_Cross']  = (df['EMA_20'] > df['EMA_50']).astype(int)
cross_change     = df['EMA_Cross'].diff().fillna(0)
golden_cross     = (cross_change == 1).sum()
death_cross      = (cross_change == -1).sum()

print('✓ EMA_50 calculated — exact Zerodha match')
print(f'  Formula : alpha × Close + (1-alpha) × prev_EMA')
print(f'  alpha   : 2 / (50+1) = {2/51:.6f}')
print(f'  Current : {df["EMA_50"].iloc[-1]}')
print(f'  Golden Cross events : {golden_cross}')
print(f'  Death Cross events  : {death_cross}')
print(f'\n  Sample (last 5 rows):')
print(df[['Date', 'Close', 'EMA_20', 'EMA_50', 'EMA_Cross']].tail(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — RSI 14 (Exact Zerodha match)
#
# TradingView ta.rsi() uses Wilder's RMA (not simple SMA, not ewm alpha)
# Wilder's RMA formula:
#   rma_t = (rma_(t-1) × (period-1) + value_t) / period
#   which is equivalent to: alpha = 1/period, ewm with adjust=False
#
# The "Smoothing Line SMA(14)" in Zerodha is a SEPARATE signal line —
# it does NOT change the RSI value itself.
#
# Step 1: delta    = Close_t − Close_(t-1)
# Step 2: gain     = max(delta, 0)
#          loss     = max(-delta, 0)
# Step 3: avg_gain = Wilder's RMA of gain over 14 periods
#          avg_loss = Wilder's RMA of loss over 14 periods
# Step 4: RS  = avg_gain / avg_loss
# Step 5: RSI = 100 − (100 / (1 + RS))
# ─────────────────────────────────────────────────────────────────────────
def calc_rma(series, period):
    """
    Wilder's RMA — matches TradingView ta.rma() exactly.
    rma_t = (rma_(t-1) × (period-1) + value_t) / period
    Seed  : first rma = simple mean of first `period` values
    """
    result = [None] * len(series)
    values = series.values
    # Seed: average of first `period` values
    result[period - 1] = np.mean(values[:period])
    for i in range(period, len(values)):
        result[i] = (result[i - 1] * (period - 1) + values[i]) / period
    return pd.Series(result, index=series.index)

RSI_PERIOD = 14

delta    = df['Close'].diff()
gain     = delta.clip(lower=0)
loss     = (-delta).clip(lower=0)

avg_gain = calc_rma(gain.fillna(0), RSI_PERIOD)
avg_loss = calc_rma(loss.fillna(0), RSI_PERIOD)

rs           = avg_gain / avg_loss
df['RSI_14'] = (100 - (100 / (1 + rs))).round(2)

overbought = (df['RSI_14'] > 70).sum()
oversold   = (df['RSI_14'] < 30).sum()

print('✓ RSI_14 calculated — exact Zerodha match (Wilder RMA)')
print(f'  Method  : Wilder RMA = (prev_rma × 13 + value) / 14')
print(f'  Seed    : SMA of first 14 gains/losses')
print(f'  Current : {df["RSI_14"].iloc[-1]}')
print(f'  Overbought (>70) : {overbought} days')
print(f'  Oversold   (<30) : {oversold} days')
print(f'\n  Sample (last 5 rows):')
print(df[['Date', 'Close', 'RSI_14']].tail(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Bollinger Bands (Exact Zerodha match)
#
# Zerodha settings: Length=20, Mult=2, Offset=0
# Middle : SMA(close, 20) = simple rolling mean
# Upper  : Middle + 2 × rolling std(20)
# Lower  : Middle − 2 × rolling std(20)
# TradingView uses population std (ddof=0) by default
# ─────────────────────────────────────────────────────────────────────────
BB_PERIOD = 20
BB_MULT   = 2

rolling_mean     = df['Close'].rolling(window=BB_PERIOD).mean()
rolling_std      = df['Close'].rolling(window=BB_PERIOD).std(ddof=0)  # population std — matches TradingView

df['BB_Upper']   = (rolling_mean + BB_MULT * rolling_std).round(2)
df['BB_Middle']  = rolling_mean.round(2)
df['BB_Lower']   = (rolling_mean - BB_MULT * rolling_std).round(2)
df['BB_Width']   = (df['BB_Upper'] - df['BB_Lower']).round(2)
df['BB_Position']= (
    (df['Close'] - df['BB_Lower']) /
    (df['BB_Upper'] - df['BB_Lower'])
).round(4)

latest = df.iloc[-1]
print('✓ Bollinger Bands calculated — exact Zerodha match')
print(f'  Period              : {BB_PERIOD} days')
print(f'  Multiplier          : {BB_MULT}')
print(f'  Std type            : population std (ddof=0) — matches TradingView')
print(f'  Current values:')
print(f'    Close      : {latest["Close"]:,.2f}')
print(f'    BB_Upper   : {latest["BB_Upper"]:,.2f}')
print(f'    BB_Middle  : {latest["BB_Middle"]:,.2f}')
print(f'    BB_Lower   : {latest["BB_Lower"]:,.2f}')
print(f'\n  Sample (last 5 rows):')
print(df[['Date','Close','BB_Upper','BB_Middle','BB_Lower']].tail(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — MACD Line (Exact Zerodha match)
#
# Zerodha settings: Fast=12, Slow=26, Oscillator MA Type=EMA
# MACD Line = EMA(close, 12) − EMA(close, 26)
# Uses same EMA formula as calc_ema() above — matches Zerodha exactly
# ─────────────────────────────────────────────────────────────────────────
MACD_FAST = 12
MACD_SLOW = 26

ema_fast        = calc_ema(df['Close'], MACD_FAST)
ema_slow        = calc_ema(df['Close'], MACD_SLOW)
df['MACD_Line'] = (ema_fast - ema_slow).round(4)

print('✓ MACD_Line calculated — exact Zerodha match')
print(f'  Formula  : EMA({MACD_FAST}) − EMA({MACD_SLOW})')
print(f'  Current  : {df["MACD_Line"].iloc[-1]}')
print(f'\n  Sample (last 5 rows):')
print(df[['Date', 'Close', 'MACD_Line']].tail(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Signal Line and MACD Histogram (Exact Zerodha match)
#
# Zerodha settings: Signal Length=9, Signal Line MA Type=EMA
# Signal Line = EMA(MACD_Line, 9)
# Histogram   = MACD_Line − Signal_Line
# ─────────────────────────────────────────────────────────────────────────
MACD_SIGNAL = 9

df['MACD_Signal'] = calc_ema(df['MACD_Line'], MACD_SIGNAL).round(4)
df['MACD_Hist']   = (df['MACD_Line'] - df['MACD_Signal']).round(4)

macd_bull = ((df['MACD_Line'] > df['MACD_Signal']) &
             (df['MACD_Line'].shift(1) <= df['MACD_Signal'].shift(1))).sum()
macd_bear = ((df['MACD_Line'] < df['MACD_Signal']) &
             (df['MACD_Line'].shift(1) >= df['MACD_Signal'].shift(1))).sum()

print('✓ MACD Signal and Histogram calculated — exact Zerodha match')
print(f'  Signal   : EMA({MACD_SIGNAL}) of MACD Line')
print(f'  Histogram: MACD Line − Signal Line')
print(f'  Bullish crossovers : {macd_bull}')
print(f'  Bearish crossovers : {macd_bear}')
print(f'  Current MACD Line  : {df["MACD_Line"].iloc[-1]}')
print(f'  Current Signal     : {df["MACD_Signal"].iloc[-1]}')
print(f'  Current Histogram  : {df["MACD_Hist"].iloc[-1]}')
print(f'\n  Sample (last 5 rows):')
print(df[['Date','MACD_Line','MACD_Signal','MACD_Hist']].tail(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 9 — Full Dataset Preview and Signal Summary
# ─────────────────────────────────────────────────────────────────────────
print('=' * 75)
print('  COMPLETE INDICATOR DATASET — EXACT ZERODHA MATCH')
print('=' * 75)
print(f'  Rows    : {len(df)} trading days')
print(f'  Columns : {len(df.columns)}')
print(f'  From    : {df["Date"].iloc[0].date()}')
print(f'  To      : {df["Date"].iloc[-1].date()}')

print('\n  All columns:')
for col in df.columns:
    print(f'    {col:<20} : {df[col].notna().sum()} non-null values')

print('\n  Last 5 rows:')
display_cols = [
    'Date','Open','High','Low','Close',
    'EMA_20','EMA_50',
    'RSI_14',
    'BB_Upper','BB_Middle','BB_Lower',
    'MACD_Line','MACD_Signal','MACD_Hist'
]
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(df[display_cols].tail(5).to_string(index=False))

latest = df.iloc[-1]
print('\n  Signal Summary (latest day):')
print(f'    EMA Cross : {"Bullish (EMA20>EMA50)" if latest["EMA_Cross"]==1 else "Bearish (EMA20<EMA50)"}')
print(f'    RSI       : {latest["RSI_14"]} — {"Overbought" if latest["RSI_14"]>70 else ("Oversold" if latest["RSI_14"]<30 else "Neutral")}')
print(f'    MACD      : {"Bullish" if latest["MACD_Hist"]>0 else "Bearish"} (Hist={latest["MACD_Hist"]})')
print(f'    BB Pos    : {latest["BB_Position"]} — {"Near upper" if latest["BB_Position"]>0.8 else ("Near lower" if latest["BB_Position"]<0.2 else "Mid-range")}')

print('\n  Methods used (Zerodha matching):')
print('    EMA 20/50 : alpha×Close + (1-alpha)×prev_EMA, seed=first Close')
print('    RSI 14    : Wilder RMA = (prev×13 + val)/14, seed=SMA(14)')
print('    BB        : SMA(20) ± 2×population_std(20)')
print('    MACD      : EMA(12)−EMA(26), Signal=EMA(9) of MACD')

## ✅ Cell 10 — Zerodha Verification Tool

**How to use:**
1. Open Zerodha Kite → Charts → pick any date → hover over the candle
2. Note the values shown for EMA20, EMA50, RSI, BB Upper/Middle/Lower, MACD
3. Enter that date and the Zerodha values in the `ZERODHA_VALUES` dictionary below
4. Run the cell — it will compare your calculated values vs Zerodha side by side

**Example:** If Zerodha shows on 15-Jan-2025: EMA20=23450.10, RSI=58.32 → enter those below

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 10 — ZERODHA VERIFICATION TOOL
#
# INSTRUCTIONS:
# 1. Open Zerodha Kite → Nifty 50 chart → daily timeframe
# 2. Add indicators: EMA(20), EMA(50), RSI(14), BB(20,2), MACD(12,26,9)
# 3. Hover over any candle and note the values
# 4. Enter the date and Zerodha values below
# 5. Run this cell to see side-by-side comparison
# ─────────────────────────────────────────────────────────────────────────

# ── ENTER YOUR ZERODHA VALUES HERE ────────────────────────────────────────
# Change the date to any date you want to verify (must exist in your CSV)
VERIFY_DATE = '28-Apr-2026'   # ← change to your chosen date (dd-Mon-YYYY)

ZERODHA_VALUES = {
    'EMA_20'     : None,    # ← enter Zerodha EMA20 value here   e.g. 23450.10
    'EMA_50'     : None,    # ← enter Zerodha EMA50 value here   e.g. 23200.50
    'RSI_14'     : None,    # ← enter Zerodha RSI14 value here   e.g. 58.32
    'BB_Upper'   : None,    # ← enter Zerodha BB Upper value here e.g. 24100.00
    'BB_Middle'  : None,    # ← enter Zerodha BB Middle value here e.g. 23500.00
    'BB_Lower'   : None,    # ← enter Zerodha BB Lower value here e.g. 22900.00
    'MACD_Line'  : None,    # ← enter Zerodha MACD Line value here e.g. 125.40
    'MACD_Signal': None,    # ← enter Zerodha Signal Line value here e.g. 118.20
    'MACD_Hist'  : None,    # ← enter Zerodha Histogram value here  e.g. 7.20
}
# ──────────────────────────────────────────────────────────────────────────

# Parse the verification date
try:
    verify_dt = pd.to_datetime(VERIFY_DATE, dayfirst=True)
    row = df[df['Date'].dt.date == verify_dt.date()]

    if row.empty:
        print(f'⚠️  Date {VERIFY_DATE} not found in your dataset.')
        print(f'   Available dates range: {df["Date"].iloc[0].date()} → {df["Date"].iloc[-1].date()}')
        print('   Please change VERIFY_DATE to a valid trading day.')
    else:
        row = row.iloc[0]

        print('=' * 70)
        print(f'  ZERODHA VERIFICATION — {VERIFY_DATE}')
        print(f'  Close price on this date: {row["Close"]:,.2f}')
        print('=' * 70)
        print(f'  {"Indicator":<15} {"Our Value":>14} {"Zerodha Value":>15} {"Difference":>12} {"Match?":>8}')
        print('  ' + '─' * 66)

        indicators = [
            ('EMA_20',      'EMA_20'),
            ('EMA_50',      'EMA_50'),
            ('RSI_14',      'RSI_14'),
            ('BB_Upper',    'BB_Upper'),
            ('BB_Middle',   'BB_Middle'),
            ('BB_Lower',    'BB_Lower'),
            ('MACD_Line',   'MACD_Line'),
            ('MACD_Signal', 'MACD_Signal'),
            ('MACD_Hist',   'MACD_Hist'),
        ]

        all_match = True
        for label, col in indicators:
            our_val      = row[col]
            zerodha_val  = ZERODHA_VALUES.get(col)

            if our_val is None or (isinstance(our_val, float) and pd.isna(our_val)):
                print(f'  {label:<15} {"N/A":>14} {"—":>15} {"—":>12} {"—":>8}')
                continue

            our_str = f'{our_val:,.4f}'

            if zerodha_val is None:
                print(f'  {label:<15} {our_str:>14} {"(not entered)":>15} {"—":>12} {"—":>8}')
            else:
                diff     = abs(float(our_val) - float(zerodha_val))
                pct_diff = (diff / abs(float(zerodha_val)) * 100) if zerodha_val != 0 else 0
                match    = '✅ YES' if pct_diff < 0.1 else ('⚠️  CLOSE' if pct_diff < 0.5 else '❌ NO')
                if '❌' in match:
                    all_match = False
                z_str    = f'{float(zerodha_val):,.4f}'
                d_str    = f'{diff:,.4f}'
                print(f'  {label:<15} {our_str:>14} {z_str:>15} {d_str:>12} {match:>8}')

        print('  ' + '─' * 66)

        # Check if any values were actually entered
        entered = [v for v in ZERODHA_VALUES.values() if v is not None]
        if not entered:
            print('\n  ℹ️  No Zerodha values entered yet.')
            print('  Enter values from Zerodha Kite in ZERODHA_VALUES above and re-run.')
        else:
            print(f'\n  Match Threshold: < 0.1% difference = ✅ YES')
            print(f'                   0.1% to 0.5%     = ⚠️  CLOSE (rounding difference)')
            print(f'                   > 0.5% difference = ❌ NO (formula mismatch)')
            if all_match:
                print('\n  🎉 All entered values match Zerodha!')
            else:
                print('\n  ⚠️  Some values do not match — check the ❌ rows above.')
                print('  Possible reasons:')
                print('    1. Date mismatch — Zerodha may show a different candle date')
                print('    2. Rounding — Zerodha rounds to 2 decimal places')
                print('    3. History length — more historical data improves EMA accuracy')

        print('\n  Our calculated values on this date:')
        check_cols = ['Date','Close','EMA_20','EMA_50','RSI_14',
                      'BB_Upper','BB_Middle','BB_Lower','MACD_Line','MACD_Signal','MACD_Hist']
        print(df.loc[df['Date'].dt.date == verify_dt.date(), check_cols].to_string(index=False))

except Exception as e:
    print(f'Error: {e}')
    print('Please check the VERIFY_DATE format — use dd-Mon-YYYY e.g. 28-Apr-2026')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 11 — Save Final CSV and Download
# ─────────────────────────────────────────────────────────────────────────
OUTPUT_FILE = 'nifty50_with_indicators_zerodha.csv'

df_save = df.copy()
df_save['Date'] = df_save['Date'].dt.strftime('%d-%b-%Y')

final_cols = [
    'Date','Open','High','Low','Close',
    'EMA_20','EMA_50','EMA_Cross',
    'RSI_14',
    'BB_Upper','BB_Middle','BB_Lower','BB_Width','BB_Position',
    'MACD_Line','MACD_Signal','MACD_Hist'
]
df_save = df_save[final_cols]
df_save.to_csv(OUTPUT_FILE, index=False)

print('=' * 60)
print(f'  ✓ File          : {OUTPUT_FILE}')
print(f'  ✓ Rows          : {len(df_save)}')
print(f'  ✓ Columns       : {len(df_save.columns)}')
print('  ✓ Columns saved :')
for col in df_save.columns:
    print(f'      {col}')
print('=' * 60)
print('\n⬇️  Downloading...')
files.download(OUTPUT_FILE)
print('✓ Done! Values should now match Zerodha exactly.')
print('✅ Step 2 Complete — Ready for Step 3 (Log-Return Transformation)')